# V1-02: 재무 조회·계산 개발 기준선

이 notebook은 V1의 재무 데이터 수집과 결정적 계산이 정해 둔 소규모 사례에서 정확한지 확인합니다. LLM, LangChain, LangGraph, RAG는 사용하지 않습니다.

측정 흐름은 다음과 같습니다.

`고정 OpenDART 수치 → client.py 조회 결과 → Field Accuracy → Python 계산 결과 → Calculation Accuracy`

## 0. 이 기준선의 범위와 한계

- 사례는 삼성전자, SK하이닉스, 현대자동차의 2023·2024 사업보고서 연결재무제표입니다.
- 기대 수치는 2026-09-02에 OpenDART raw 응답에서 확인한 개발용 snapshot입니다. 각 사례에는 `corp_code`, `rcept_no`, 보고서·재무제표 기준, 계정 식별자를 함께 남깁니다.
- 이 결과는 현재 추출 코드의 회귀를 확인하는 작은 기준선이며, V1 최종 평가셋이나 모델 성능 비교가 아닙니다.
- 별도재무제표 대체(`OFS`)와 API 실패 처리는 아직 이 기준선의 대상이 아닙니다.

## 1. 환경과 기존 재무 조회 함수 불러오기

재무 추출 로직을 notebook에 다시 작성하지 않고, 현재 검증 대상인 `src/dart/client.py`의 `get_financial_summary()`만 사용합니다.

In [ ]:
import os
import sys
from pathlib import Path

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == "v1":
    project_root = current_dir.parents[1]
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"{env_path} 파일을 먼저 만들어 주세요.")

for raw_line in env_path.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))

api_key = os.environ.get("OPENDART_API_KEY")
if not api_key:
    raise ValueError(".env에 OPENDART_API_KEY를 설정해 주세요.")

print("OpenDART API 키를 읽었습니다. 값은 출력하지 않습니다.")

In [ ]:
sys.path.insert(0, str(project_root / "src"))

from dart.client import get_financial_summary

## 2. 개발용 정답 snapshot

각 사례의 `expected`는 OpenDART 사업보고서(`11011`)·연결재무제표(`CFS`) raw 응답에서 확인한 원본 문자열입니다. 금액을 임의로 반올림하거나 단위를 바꾸지 않습니다.

In [ ]:
SNAPSHOT_DATE = "2026-09-02"
FINANCIAL_FIELDS = ["매출액", "영업이익", "당기순이익"]

financial_cases = [
    {
        "기업명": "삼성전자",
        "사업연도": 2023,
        "provenance": {"corp_code": "00126380", "rcept_no": "20240312000736", "report_code": "11011", "fs_div": "CFS", "account_ids": {"매출액": "ifrs-full_Revenue", "영업이익": "dart_OperatingIncomeLoss", "당기순이익": "ifrs-full_ProfitLoss"}},
        "expected": {
            "매출액": "258935494000000",
            "영업이익": "6566976000000",
            "당기순이익": "15487100000000",
        },
    },
    {
        "기업명": "삼성전자",
        "사업연도": 2024,
        "provenance": {"corp_code": "00126380", "rcept_no": "20250311001085", "report_code": "11011", "fs_div": "CFS", "account_ids": {"매출액": "ifrs-full_Revenue", "영업이익": "dart_OperatingIncomeLoss", "당기순이익": "ifrs-full_ProfitLoss"}},
        "expected": {
            "매출액": "300870903000000",
            "영업이익": "32725961000000",
            "당기순이익": "34451351000000",
        },
    },
    {
        "기업명": "SK하이닉스",
        "사업연도": 2023,
        "provenance": {"corp_code": "00164779", "rcept_no": "20240319000684", "report_code": "11011", "fs_div": "CFS", "account_ids": {"매출액": "ifrs-full_Revenue", "영업이익": "dart_OperatingIncomeLoss", "당기순이익": "ifrs-full_ProfitLoss"}},
        "expected": {
            "매출액": "32765719000000",
            "영업이익": "-7730313000000",
            "당기순이익": "-9137547000000",
        },
    },
    {
        "기업명": "SK하이닉스",
        "사업연도": 2024,
        "provenance": {"corp_code": "00164779", "rcept_no": "20250319000665", "report_code": "11011", "fs_div": "CFS", "account_ids": {"매출액": "ifrs-full_Revenue", "영업이익": "dart_OperatingIncomeLoss", "당기순이익": "ifrs-full_ProfitLoss"}},
        "expected": {
            "매출액": "66192960000000",
            "영업이익": "23467319000000",
            "당기순이익": "19796902000000",
        },
    },
    {
        "기업명": "현대자동차",
        "사업연도": 2023,
        "provenance": {"corp_code": "00164742", "rcept_no": "20240313001451", "report_code": "11011", "fs_div": "CFS", "account_ids": {"매출액": "ifrs-full_Revenue", "영업이익": "dart_OperatingIncomeLoss", "당기순이익": "ifrs-full_ProfitLoss"}},
        "expected": {
            "매출액": "162663579000000",
            "영업이익": "15126901000000",
            "당기순이익": "12272301000000",
        },
    },
    {
        "기업명": "현대자동차",
        "사업연도": 2024,
        "provenance": {"corp_code": "00164742", "rcept_no": "20250312001148", "report_code": "11011", "fs_div": "CFS", "account_ids": {"매출액": "ifrs-full_Revenue", "영업이익": "dart_OperatingIncomeLoss", "당기순이익": "ifrs-full_ProfitLoss"}},
        "expected": {
            "매출액": "175231153000000",
            "영업이익": "14239592000000",
            "당기순이익": "13229908000000",
        },
    },
]

print(f"기준일: {SNAPSHOT_DATE}")
print(f"사례 수: {len(financial_cases)}개")

In [ ]:
[
    {
        "기업명": case["기업명"],
        "사업연도": case["사업연도"],
        **case["provenance"],
    }
    for case in financial_cases
]

## 3. 현재 client.py 결과 조회

이 셀은 모든 사례에 같은 함수와 사업보고서·연결 기준을 적용합니다. 결과가 snapshot과 달라지면 다음 셀에서 어떤 기업·연도·필드가 다른지 확인할 수 있습니다.

In [ ]:
actual_results = []

for case in financial_cases:
    actual = get_financial_summary(
        case["기업명"],
        case["사업연도"],
        api_key,
        report_code="11011",
        fs_div="CFS",
    )
    actual_results.append({**case, "actual": actual})

actual_results

## 4. Field Accuracy 계산

Field Accuracy는 비교한 모든 재무 필드 중 snapshot과 완전히 일치한 필드의 비율입니다. 이 notebook에서는 `6개 사례 × 3개 필드 = 18개`를 비교합니다.

In [ ]:
field_results = []

for result in actual_results:
    for field in FINANCIAL_FIELDS:
        expected = result["expected"][field]
        actual = result["actual"][field]
        field_results.append(
            {
                "기업명": result["기업명"],
                "사업연도": result["사업연도"],
                "필드": field,
                "일치": actual == expected,
                "expected": expected,
                "actual": actual,
            }
        )

field_results

In [8]:
field_accuracy = sum(row["일치"] for row in field_results) / len(field_results)
mismatched_fields = [row for row in field_results if not row["일치"]]

print(f"Field Accuracy: {field_accuracy:.1%}")
print(f"불일치 필드 수: {len(mismatched_fields)}")
mismatched_fields

Field Accuracy: 100.0%
불일치 필드 수: 0


[]

## 5. 계산 정답과 공식

- 매출 성장률 = `(현재 연도 매출액 / 이전 연도 매출액 - 1) × 100`
- 영업이익률 = `(영업이익 / 매출액) × 100`

아래 기대값은 소수 넷째 자리까지 고정합니다. Calculator 평가는 `client.py`의 조회 결과가 아니라 위의 고정 `expected` 금액만 입력으로 사용합니다. 따라서 수치 추출 오류와 계산 오류를 분리할 수 있습니다.

In [ ]:
calculation_cases = [
    {"기업명": "삼성전자", "expected_growth": 16.1953, "expected_margin": 10.8771},
    {"기업명": "SK하이닉스", "expected_growth": 102.0189, "expected_margin": 35.4529},
    {"기업명": "현대자동차", "expected_growth": 7.7261, "expected_margin": 8.1262},
]

In [ ]:
def to_int(amount):
    return int(amount.replace(",", ""))

expected_by_key = {
    (case["기업명"], case["사업연도"]): case["expected"]
    for case in financial_cases
}

calculation_results = []
for case in calculation_cases:
    previous = expected_by_key[(case["기업명"], 2023)]
    current = expected_by_key[(case["기업명"], 2024)]

    growth = (to_int(current["매출액"]) / to_int(previous["매출액"]) - 1) * 100
    margin = to_int(current["영업이익"]) / to_int(current["매출액"]) * 100

    calculation_results.append(
        {
            "기업명": case["기업명"],
            "매출 성장률(%)": round(growth, 4),
            "영업이익률(%)": round(margin, 4),
            "성장률 일치": round(growth, 4) == case["expected_growth"],
            "이익률 일치": round(margin, 4) == case["expected_margin"],
        }
    )

calculation_results

## 6. Calculation Accuracy 계산

Calculation Accuracy는 비교한 성장률·영업이익률 6개 값 중 정확히 일치한 값의 비율입니다. 계산식이 Python 코드에 명시되어 있으므로, LLM 응답 품질과 분리해서 해석할 수 있습니다.

In [11]:
calculation_checks = [
    row[check]
    for row in calculation_results
    for check in ["성장률 일치", "이익률 일치"]
]
calculation_accuracy = sum(calculation_checks) / len(calculation_checks)

print(f"Calculation Accuracy: {calculation_accuracy:.1%}")
print(f"비교한 계산 값 수: {len(calculation_checks)}")

Calculation Accuracy: 100.0%
비교한 계산 값 수: 6


## 7. 최소 기준선 통과 여부

이 작은 개발 기준선에서는 모든 snapshot 필드·계산 결과가 일치해야 통과입니다. 불일치가 있으면 다음 기술 단계로 넘어가지 않고 `client.py`의 계정 선택과 원본 공시를 먼저 확인합니다.

In [12]:
assert field_accuracy == 1.0
assert calculation_accuracy == 1.0

print("V1 재무 조회·계산 개발 기준선을 통과했습니다.")

V1 재무 조회·계산 개발 기준선을 통과했습니다.


## 이번 notebook에서 이해할 핵심

1. Field Accuracy는 수치 추출이 맞는지를, Calculation Accuracy는 고정 snapshot 수치로 계산 공식과 결과가 맞는지를 분리해 봅니다.
2. 수치 조회와 계산은 LLM을 거치지 않아야 재현성과 오류 위치를 분명히 할 수 있습니다.
3. 이 notebook은 현재 코드의 회귀를 막는 작은 개발 기준선입니다. 기업·업종·연도를 넓힌 최종 평가셋은 별도 합의 후 만듭니다.
4. 다음 데이터 계층 검증에서는 연결재무제표가 없는 경우, 공시 목록과 원문 링크의 실패·누락 경로를 별도로 확인해야 합니다.